In [19]:
%%capture

%pip install langchain-community -U
%pip install langchain-google-genai
%pip install pypdf
%pip install chromadb
%pip install langchain
%pip install langchain-chroma
%pip install langchain -U

I0000 00:00:1754967024.027722   75841 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1754967030.550125   75841 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1754967035.443004   75841 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1754967039.459143   75841 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1754967044.540989   75841 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1754967047.798173   75841 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1754967053.999382   75841 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


In [20]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.retrievers import ParentDocumentRetriever
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from langchain.chains.combine_documents import create_stuff_documents_chain

In [21]:
# Carregar e extrair pdf
loader = PyPDFLoader('os-sertoes.pdf')
documents = loader.load()

In [22]:
# Criar chuncks
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=128)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=32)

In [23]:
embeddings_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# retriever 
vectorstore = Chroma(embedding_function=embeddings_model)

In [24]:
from langchain.storage import InMemoryStore
store = InMemoryStore()
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

In [25]:
retriever.add_documents(documents=documents, ids=None)

In [26]:
# Inicialize o modelo Gemini
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2)
# Crie o template do prompt
prompt = ChatPromptTemplate.from_template("""
    Você é um bibliotecário. Responda as perguntas baseadas no contexto fornecido.
                                          
    Context: {context}
                                          
    Pergunta: {input}
""")

document_chain = create_stuff_documents_chain(llm, prompt)

In [27]:
asks = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em \"Os Sertões\"? Como esses aspectos refletem a visão do autor sobre o Brasil da época?"
]

for ask in asks:
    # Chame a Chain com uma pergunta
    context = retriever.invoke(ask)
    response = document_chain.invoke({"input": ask, "context": context})

    # Imprima a resposta
    print(f"Pergunta: {ask}\nResposta: {response}\n\n")

Pergunta: Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?
Resposta: Euclides da Cunha descreve o ambiente natural do sertão nordestino como um espaço de contrastes extremos e imprevisibilidade,  caracterizado por um conflito perene entre fatores geológicos, topográficos e climáticos.  Ele não identifica um fator preponderante, mas sim uma interação complexa e cíclica que resulta em uma mesologia singular.  A região sofre de secas devastadoras, intercaladas por períodos de chuvas intensas, mas irregulares.  A falta de estudos científicos aprofundados sobre a região, devido à dificuldade de acesso e à indiferença, contribui para o desconhecimento de suas características.

Essa natureza inconstante e hostil molda profundamente a vida dos sertanejos.  Eles são forçados a uma constante adaptação, desenvolvendo uma resiliência e uma capacidade de reação rápida a situações adversas.  Sua vida é marcada por períodos